In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
train_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/train_dataset.csv"

train_df = pd.read_csv(train_path)

print(train_df.shape)

train_df.head()

(243925, 45)


,acc_resultant_mean,acc_resultant_std,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_horizontal_mean,acc_horizontal_std,acc_horizontal_min,acc_horizontal_max,...,pitch_rms,yaw_mean,yaw_std,yaw_min,yaw_max,yaw_median,yaw_rms,driver,road_type,behavior
0,0.058162,0.040807,0.014213,0.178804,0.047605,0.070658,0.051064,0.037351,0.001414,0.140004,...,0.010596,-0.033400,0.002472,-0.037,-0.029,-0.0335,0.033488,D6,MOTORWAY,AGGRESSIVE
1,0.054311,0.033896,0.014213,0.141287,0.047605,0.063721,0.048567,0.034008,0.001414,0.140004,...,0.011027,-0.033333,0.002591,-0.037,-0.028,-0.0335,0.033431,D6,MOTORWAY,AGGRESSIVE
2,0.055394,0.033481,0.014213,0.141287,0.050094,0.064437,0.049765,0.033695,0.001414,0.140004,...,0.011435,-0.033133,0.002909,-0.037,-0.026,-0.0335,0.033257,D6,MOTORWAY,AGGRESSIVE
3,0.053637,0.033202,0.014213,0.141287,0.047605,0.062790,0.050241,0.033256,0.001414,0.140004,...,0.011435,-0.032833,0.003354,-0.037,-0.024,-0.0335,0.032998,D6,MOTORWAY,AGGRESSIVE
4,0.051045,0.031886,0.014213,0.141287,0.044649,0.059903,0.047618,0.031777,0.001414,0.140004,...,0.011364,-0.032400,0.003865,-0.037,-0.022,-0.0325,0.032622,D6,MOTORWAY,AGGRESSIVE


In [4]:
print(f"Number of Features : {len(train_df.columns)-3}")

train_df.columns.tolist()

Number of Features : 42


['acc_resultant_mean',
 'acc_resultant_std',
 'acc_resultant_min',
 'acc_resultant_max',
 'acc_resultant_median',
 'acc_resultant_rms',
 'acc_horizontal_mean',
 'acc_horizontal_std',
 'acc_horizontal_min',
 'acc_horizontal_max',
 'acc_horizontal_median',
 'acc_horizontal_rms',
 'speed_mean',
 'speed_std',
 'speed_min',
 'speed_max',
 'speed_median',
 'speed_rms',
 'speed_delta_mean',
 'speed_delta_std',
 'speed_delta_min',
 'speed_delta_max',
 'speed_delta_median',
 'speed_delta_rms',
 'roll_mean',
 'roll_std',
 'roll_min',
 'roll_max',
 'roll_median',
 'roll_rms',
 'pitch_mean',
 'pitch_std',
 'pitch_min',
 'pitch_max',
 'pitch_median',
 'pitch_rms',
 'yaw_mean',
 'yaw_std',
 'yaw_min',
 'yaw_max',
 'yaw_median',
 'yaw_rms',
 'driver',
 'road_type',
 'behavior']

In [5]:
def load_gps(gps_path):

    gps_columns = [

        "timestamp",
        "speed",
        "latitude",
        "longitude",
        "altitude",

        "gps_quality",
        "satellites",

        "heading",

        "extra_1",
        "extra_2",
        "extra_3",
        "extra_4"

    ]

    gps_df = pd.read_csv(
        gps_path,
        sep=r"\s+",
        header=None,
        names=gps_columns
    )

    return gps_df

def synchronize_sensors(acc_df, gps_df):

    master_df = pd.merge_asof(

        acc_df.sort_values("timestamp"),

        gps_df.sort_values("timestamp"),

        on="timestamp",

        direction="nearest"

    )

    return master_df

def engineer_features(master_df):

    master_df = master_df.copy()

    # -------------------------------------------------
    # Acceleration Features
    # -------------------------------------------------

    master_df["acc_resultant"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2 +
        master_df["acc_z"]**2
    )

    master_df["acc_horizontal"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2
    )

    master_df["acc_vertical"] = master_df["acc_z"]

    # -------------------------------------------------
    # Delta Features
    # -------------------------------------------------

    master_df["speed_delta"] = master_df["speed"].diff().fillna(0)

    master_df["heading_delta"] = master_df["heading"].diff().fillna(0)

    master_df["roll_delta"] = master_df["roll"].diff().fillna(0)

    master_df["pitch_delta"] = master_df["pitch"].diff().fillna(0)

    master_df["yaw_delta"] = master_df["yaw"].diff().fillna(0)

    return master_df

def extract_statistics(signal):

    features = {}

    features["mean"] = signal.mean()

    features["std"] = signal.std()

    features["min"] = signal.min()

    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal**2)
    )

    return features

def extract_window_features(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        stats = extract_statistics(window[feature])

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

    return window_stats

def create_sliding_windows(
        feature_df,
        feature_list,
        window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features(
            window,
            feature_list
        )

        all_window_features.append(window_stats)

    return pd.DataFrame(all_window_features)

def load_accelerometer(acc_path):

    acc_columns = [

        "timestamp",
        "active",

        "acc_x",
        "acc_y",
        "acc_z",

        "acc_x_kf",
        "acc_y_kf",
        "acc_z_kf",

        "roll",
        "pitch",
        "yaw"

    ]

    acc_df = pd.read_csv(
        acc_path,
        sep=r"\s+",
        header=None,
        names=acc_columns
    )

    return acc_df

In [6]:
def extract_statistics_v2(signal):

    features = {}

    # ------------------------------
    # Basic Statistics
    # ------------------------------

    features["mean"] = signal.mean()
    features["std"] = signal.std()
    features["variance"] = signal.var()

    features["min"] = signal.min()
    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal ** 2)
    )

    # ------------------------------
    # Distribution Shape
    # ------------------------------

    features["skewness"] = signal.skew()

    features["kurtosis"] = signal.kurt()

    # ------------------------------
    # Percentiles
    # ------------------------------

    features["q25"] = signal.quantile(0.25)

    features["q75"] = signal.quantile(0.75)

    features["iqr"] = (
        features["q75"] -
        features["q25"]
    )

    return features

In [7]:
signal = train_df["speed_mean"]

extract_statistics_v2(signal)

{'mean': np.float64(94.2580538963479),
 'std': 18.228095935300402,
 'variance': 332.263481426515,
 'min': 0.0,
 'max': 148.53333333333333,
 'median': 91.73333333333336,
 'rms': np.float64(96.0043897100495),
 'skewness': np.float64(-0.5053494334147051),
 'kurtosis': np.float64(4.0567604751862305),
 'q25': np.float64(84.83666666666664),
 'q75': np.float64(104.45),
 'iqr': np.float64(19.613333333333358)}

In [8]:
def extract_window_features_v2(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        stats = extract_statistics_v2(window[feature])

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

    return window_stats

In [9]:
WINDOW_SIZE = 30

sample_window = train_df.iloc[:WINDOW_SIZE]

feature_list = [
    "speed_mean",
    "yaw_mean"
]

extract_window_features_v2(
    sample_window,
    feature_list
)

{'speed_mean_mean': np.float64(65.60944444444445),
 'speed_mean_std': 1.3212228990274244,
 'speed_mean_variance': 1.7456299489144316,
 'speed_mean_min': 63.68666666666666,
 'speed_mean_max': 67.90333333333334,
 'speed_mean_median': 65.485,
 'speed_mean_rms': np.float64(65.62230293575952),
 'speed_mean_skewness': np.float64(0.17781047890158244),
 'speed_mean_kurtosis': np.float64(-1.254726553792505),
 'speed_mean_q25': np.float64(64.47749999999999),
 'speed_mean_q75': np.float64(66.71749999999999),
 'speed_mean_iqr': np.float64(2.239999999999995),
 'yaw_mean_mean': np.float64(-0.025726666666666637),
 'yaw_mean_std': 0.00542835891036786,
 'yaw_mean_variance': 2.9467080459770144e-05,
 'yaw_mean_min': -0.0334,
 'yaw_mean_max': -0.0167,
 'yaw_mean_median': -0.0258833333333333,
 'yaw_mean_rms': np.float64(0.02627444047400859),
 'yaw_mean_skewness': np.float64(0.08337990290345279),
 'yaw_mean_kurtosis': np.float64(-1.346193773268065),
 'yaw_mean_q25': np.float64(-0.030608333333333272),
 'yaw_

In [10]:
def create_sliding_windows_v2(
    feature_df,
    feature_list,
    window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features_v2(
            window,
            feature_list
        )

        all_window_features.append(
            window_stats
        )

    return pd.DataFrame(
        all_window_features
    )

In [11]:
window_dataset_v2 = create_sliding_windows_v2(
    train_df,
    feature_list,
    WINDOW_SIZE
)

window_dataset_v2.head()

,speed_mean_mean,speed_mean_std,speed_mean_variance,speed_mean_min,speed_mean_max,speed_mean_median,speed_mean_rms,speed_mean_skewness,speed_mean_kurtosis,speed_mean_q25,...,yaw_mean_variance,yaw_mean_min,yaw_mean_max,yaw_mean_median,yaw_mean_rms,yaw_mean_skewness,yaw_mean_kurtosis,yaw_mean_q25,yaw_mean_q75,yaw_mean_iqr
0,65.609444,1.321223,1.745630,63.686667,67.903333,65.485,65.622303,0.177810,-1.254727,64.477500,...,0.000029,-0.033400,-0.016700,-0.025883,0.026274,0.083380,-1.346194,-0.030608,-0.021083,0.009525
1,65.755778,1.343836,1.805896,63.773333,68.076667,65.655,65.769051,0.154403,-1.252146,64.614167,...,0.000030,-0.033333,-0.016100,-0.025167,0.025725,0.039810,-1.316245,-0.029975,-0.020508,0.009467
2,65.905000,1.364501,1.861863,63.860000,68.250000,65.825,65.918653,0.133491,-1.246529,64.750833,...,0.000031,-0.033133,-0.015600,-0.024433,0.025157,0.002251,-1.291353,-0.029333,-0.019967,0.009367
3,66.057111,1.382899,1.912409,63.946667,68.423333,65.995,66.071103,0.115797,-1.239919,64.887500,...,0.000031,-0.032833,-0.015133,-0.023733,0.024575,-0.030569,-1.272517,-0.028675,-0.019408,0.009267
4,66.212111,1.398700,1.956361,64.033333,68.596667,66.165,66.226391,0.102264,-1.234853,65.024167,...,0.000031,-0.032400,-0.014700,-0.023083,0.023983,-0.059470,-1.260715,-0.028042,-0.018792,0.009250


In [12]:
print(window_dataset_v2.shape)

window_dataset_v2.info()

(243896, 24)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 243896 entries, 0 to 243895
Data columns (total 24 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   speed_mean_mean      243896 non-null  float64
 1   speed_mean_std       243896 non-null  float64
 2   speed_mean_variance  243896 non-null  float64
 3   speed_mean_min       243896 non-null  float64
 4   speed_mean_max       243896 non-null  float64
 5   speed_mean_median    243896 non-null  float64
 6   speed_mean_rms       243896 non-null  float64
 7   speed_mean_skewness  243896 non-null  float64
 8   speed_mean_kurtosis  243896 non-null  float64
 9   speed_mean_q25       243896 non-null  float64
 10  speed_mean_q75       243896 non-null  float64
 11  speed_mean_iqr       243896 non-null  float64
 12  yaw_mean_mean        243896 non-null  float64
 13  yaw_mean_std         243896 non-null  float64
 14  yaw_mean_variance    243896 non-null  float64
 15  yaw_

In [13]:
window_dataset_v2.describe().T.head(10)

,count,mean,std,min,25%,50%,75%,max
speed_mean_mean,243896.0,94.260562,18.155335,0.000000,84.858306,91.743333,104.429250,148.172778
speed_mean_std,243896.0,0.727914,1.478587,0.000000,0.225937,0.453658,0.849939,57.124100
speed_mean_variance,243896.0,2.716068,40.828557,0.000000,0.051047,0.205806,0.722396,3263.162831
speed_mean_min,243896.0,93.104936,18.537122,0.000000,83.880000,90.913333,103.244167,147.673333
speed_mean_max,243896.0,95.404441,17.888471,0.000000,85.760000,92.620000,105.460000,148.533333
speed_mean_median,243896.0,94.263904,18.213154,0.000000,84.853333,91.741667,104.457083,148.231667
speed_mean_rms,243896.0,94.290389,18.072573,0.000000,84.863458,91.747594,104.434190,148.173010
speed_mean_skewness,243896.0,-0.000270,0.659228,-5.477226,-0.404701,0.000000,0.399718,5.477226
speed_mean_kurtosis,243896.0,-0.745378,1.050129,-2.148042,-1.250650,-1.053013,-0.529740,30.000000
speed_mean_q25,243896.0,93.681401,18.368555,0.000000,84.373333,91.314167,103.786875,147.958333


In [14]:
def parse_trip_info(trip_name):

    parts = trip_name.split("-")

    return {
        "date": parts[0],
        "distance": parts[1],
        "driver": parts[2],
        "behavior": parts[3],
        "road_type": parts[4]
    }

In [15]:
def simplify_behavior(label):

    if "NORMAL" in label:
        return "NORMAL"

    if "AGGRESSIVE" in label:
        return "AGGRESSIVE"

    if "DROWSY" in label:
        return "DROWSY"

    return label

In [16]:
def process_trip_v2(
    dataset_path,
    driver,
    trip
):

    trip_path = os.path.join(
        dataset_path,
        driver,
        trip
    )

    acc_path = os.path.join(
        trip_path,
        "RAW_ACCELEROMETERS.txt"
    )

    gps_path = os.path.join(
        trip_path,
        "RAW_GPS.txt"
    )

    # Load Sensors
    acc_df = load_accelerometer(acc_path)
    gps_df = load_gps(gps_path)

    # Synchronize
    master_df = synchronize_sensors(
        acc_df,
        gps_df
    )

    # Feature Engineering
    feature_df = engineer_features(master_df)

    # Advanced Sliding Window
    window_dataset = create_sliding_windows_v2(
        feature_df,
        window_features,
        WINDOW_SIZE
    )

    # Labels
    info = parse_trip_info(trip)

    window_dataset["driver"] = info["driver"]
    window_dataset["road_type"] = info["road_type"]
    window_dataset["behavior"] = simplify_behavior(
        info["behavior"]
    )

    return window_dataset